# Lesson 3, Part 1

Load turtle packages and some helper funcions

In [ ]:
%pip install -q ipycc
from ipycc.turtle import Turtle, showscreen, clearscreen
def linear_spacing(xmin,xmax,num):
    interval = (xmax - xmin) / (num - 1)
    return [xmin + i * interval for i in range(num)]

def cubic(x,a,b,c,d):
    if type(x)==int or type(x)==float: return (a*x*x + b*x + c) * x + d
    else: return [(a*val*val + b*val + c) * val + d for val in x]

def rescale(x, width=390):
    xmin,xmax = min(min(x),0),max(max(x),0)
    mult = width / (xmax - xmin)
    shift = -mult * (xmax + xmin) / 2
    x_rescaled = [mult * val + shift for val in x]
    return x_rescaled,shift,mult
        
def plot_xy(x, y, pen_color='black'):
    t = Turtle()
    t.speed('fastest')
    t.penup()
    t.goto(x[0], y[0])
    t.color(pen_color)
    t.pendown()
    for i in range(1, len(x)):
        t.goto(x[i], y[i])
    t.hideturtle()        # line added to remove turtle from display

Let's chart a cubic function and some secants. 
Unfortunately, this code identifies points by their indices not their values.
Hence it is hard to guess where they are and the positions change with their spacing.

In [ ]:
def connect_points(x, y, i, j, pen_color='black'):
    plot_xy((x[i],x[j]), (y[i],y[j]), pen_color)
    
# draw quadratic and some secants
showscreen()
clearscreen()
xmin,xmax,num = 0,10,100  
a,b,c,d = 0,-1,8,10
# the curve will look similar with different num
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)
xnew,shift_x,mult_x = rescale(x)
ynew,shift_y,mult_y = rescale(y)
plot_xy(xnew,ynew)
connect_points(xnew,ynew,int(num/4),int(num/1.5),'red')
connect_points(xnew,ynew,int(num/3),int(num/1.3),'orange')
connect_points(xnew,ynew,int(num/7),int(num/2),'green')

Next let's draw secants between a fixed point (*x,y*) 
and (*x*+*dx,y*+*dy*) for various *dx*,
coloring them red for *dx* < 0 and orange for *dx* >0.

In [ ]:
def secant_cubic(x_val,nudge,a,b,c,d):
    y_val = cubic(x_val,a,b,c,d)
    y_nudged = cubic(x_val+nudge,a,b,c,d)
    secant_x = (x_val,x_val+nudge)
    secant_y = (y_val,y_nudged)
    return secant_x,secant_y
    
def draw_line(x,y,shift_x,mult_x,shift_y,mult_y,pen_color=None):
    if pen_color == None:
        pen_color = 'orange' if (x[1]>x[0]) else 'red'
    xnew = [shift_x + mult_x * val for val in x]
    ynew = [shift_y + mult_y * val for val in y]
    plot_xy(xnew,ynew,pen_color)

# draw quadratic and some secants
showscreen()
clearscreen()
xmin,xmax,num = 0,10,100  
a,b,c,d = 0,-1,8,10
x_val = 3
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)
xnew,shift_x,mult_x = rescale(x)
ynew,shift_y,mult_y = rescale(y)
plot_xy(xnew,ynew)
for nudge in linear_spacing(-3,3,8):
    sec_x,sec_y = secant_cubic(x_val,nudge,a,b,c,d)
    draw_line(sec_x,sec_y,shift_x,mult_x,shift_y,mult_y)

In the previous example, the slopes decline as we move from red to orange. 
In this example, the pattern is reversed.

In [ ]:
showscreen()
clearscreen()
xmin,xmax,num = 0,10,100 
a,b,c,d = 1,-6,-2,0
x_val = 6
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)
xnew,shift_x,mult_x = rescale(x)
ynew,shift_y,mult_y = rescale(y)
plot_xy(xnew,ynew)
for nudge in linear_spacing(-3,3,8):
    sec_x,sec_y = secant_cubic(x_val,nudge,a,b,c,d)
    draw_line(sec_x,sec_y,shift_x,mult_x,shift_y,mult_y)

The slopes converge as *dx* shrinks. 
At any particular (*x,y*), the gap between slopes for 
small positive and negative *dx* is nearly proportional to *dx*.

In [ ]:
def slope_secant(x_val,nudge,x,a,b,c,d):
    y_val = cubic(x_val,a,b,c,d)
    y_nudged = cubic(x_val+nudge,a,b,c,d)
    return (y_nudged - y_val) / nudge

xmin,xmax,num = 0,10,100 
a,b,c,d = 1,-6,-2,0
x_val,nudge = 6,10
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)

for iter in range(5):
    nudge /= 10
    slope = slope_secant(x_val,nudge,x,a,b,c,d)
    slope2 = slope_secant(x_val,-nudge,x,a,b,c,d)
    gap  = slope - slope2
    print('nudge',round(nudge,6),': secant slope',round(slope,4))
    print('nudge',round(-nudge,6),': secant slope',round(slope2,4),', gap',round(gap,4))

"Balanced" secants that span (*x,y*) on both sides don't change their slopes as much as *dx* shrinks.

In [ ]:
showscreen()
# draw balanced secants in previous chart
for nudge in [4,3,2,1]:
    sec_x,sec_y = secant_cubic(x_val-nudge/2,nudge,a,b,c,d)
    draw_line(sec_x,sec_y,shift_x,mult_x,shift_y,mult_y,'green')

Balanced secants around (*x,y*) converge in slope much faster than secants based at (*x,y*). 
That's what better_derivative_of_integral() in the previous lesson used for its computations.

In [ ]:
def slope_balanced_secant(x_val,nudge,a,b,c,d):
    y_minus = cubic(x_val-nudge/2,a,b,c,d)
    y_plus = cubic(x_val+nudge/2,a,b,c,d)
    return (y_plus - y_minus) / nudge

xmin,xmax,num = 0,10,300  # try again with num = 200
a,b,c,d = 1,-6,-2,0
x_val,nudge = 6,10
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)

for iter in range(5):
    nudge /= 10
    slope = slope_balanced_secant(x_val,nudge,a,b,c,d)
    print('nudge',round(nudge,6),': secant slope',round(slope,10))

If we rewrite the function to set a standard small nudge and make changes optional, 
we obtain a handy derivative calculator for cubics.

In [ ]:
def approx_deriv_cubic(x_val,a,b,c,d,nudge=0.001):  # optional arguments must go last
    y_minus = cubic(x_val-nudge/2,a,b,c,d)
    y_plus = cubic(x_val+nudge/2,a,b,c,d)
    return (y_plus - y_minus) / nudge

xmin,xmax,num = 0,10,300
a,b,c,d = 1,-6,-2,0
x_val,nudge = 6,10
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)

for x_val in range(10):
    print('x =',x_val,', derivative',round(approx_deriv_cubic(x_val,a,b,c,d,),2))

If we draw tangent lines at short intervals, their inner edge (also known as an "envelope") traces *y* = *f*(*x*) very well. 
This illustrates a crucial principle of calculus, which we first saw for circles nested inside polygons, 
namely that smooth curves can be treated as chains of very short straight lines.

In [ ]:
def tangent_line(x_val,y_val,slope,width):
    tangent_x = x_val-width/2,x_val+width/2
    height = slope * width
    tangent_y = y_val-height/2,y_val+height/2
    return tangent_x,tangent_y

showscreen()
clearscreen()
xmin,xmax,num = 0,10,100
a,b,c,d = 1,-6,-2,0
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)
xnew,shift_x,mult_x = rescale(x)
ynew,shift_y,mult_y = rescale(y)

# plot cubic and compare to envelope of its tangent lines
plot_xy(xnew,ynew)    
for x_val in linear_spacing(xmin+1,xmax-1,20):
    y_val = cubic(x_val,a,b,c,d)
    slope = approx_deriv_cubic(x_val,a,b,c,d)
    tan_x,tan_y = tangent_line(x_val,y_val,slope,width=2)
    draw_line(tan_x,tan_y,shift_x,mult_x,shift_y,mult_y,'magenta')

For a downward curving function, the tangent lines lie above it but their envelope still traces *f*(*x*).

In [ ]:
xmin,xmax,num = 0,10,100
a,b,c,d = -1,-1,50,0
x = linear_spacing(xmin,xmax,num)
y = cubic(x,a,b,c,d)
xnew,shift_x,mult_x = rescale(x)
ynew,shift_y,mult_y = rescale(y)

# plot cubic and compare to envelope of its tangent lines
showscreen()
clearscreen()
plot_xy(xnew,ynew)    
for x_val in linear_spacing(xmin+1,xmax-1,20):
    y_val = cubic(x_val,a,b,c,d)
    slope = approx_deriv_cubic(x_val,a,b,c,d)
    tan_x,tan_y = tangent_line(x_val,y_val,slope,width=2)
    draw_line(tan_x,tan_y,shift_x,mult_x,shift_y,mult_y,'magenta')

The previous lesson confirmed for a sample cubic *f* 
that the approximate derivative of its approximate integral 
is very close to *f*. 
Here we see that the approximate integral 
of its approximate derivative is also close to *f*, 
provided the integral is initialized at *f*(*x*[0]).

In [ ]:
def approx_deriv_list(xmin,xmax,a,b,c,d,nudge=0.001):
    x = linear_spacing(xmin,xmax,num=500)
    y = cubic(x,a,b,c,d)
    dydx = [approx_deriv_cubic(val,a,b,c,d,nudge) for val in x]
    return x,y,dydx

# estimate area under derivatives, shift by y[0], and compare to original function
xmin,xmax = 0,10
a,b,c,d = 1,2,3,4
x,y,dydx = approx_deriv_list(xmin,xmax,a,b,c,d)
dx = x[1] - x[0]
sum_area = y[0]
sum_abs_error = 0
for i in range(1,len(y)):
    sum_area += 0.5 * dx * (dydx[i-1] + dydx[i])
    sum_abs_error += abs(sum_area - y[i])
avg_abs_rel_err = sum_abs_error / (len(y) - 1)
print('The average absolute relative error is', round(avg_abs_rel_err,6))

Let's compare the shapes of cubic functions and their derivatives. 
Note that derivatives of zero typically mark turning points in the function.

In [ ]:
showscreen()
clearscreen()
xmin,xmax,width = 0,10,390
a,b,c,d = -1,10,-10,-10
x,y,dydx = approx_deriv_list(xmin,xmax,a,b,c,d)
# plot y and dydx
xnew,shift_x,mult_x = rescale(x)
joint,shift_joint,mult_joint = rescale(y+dydx)
ynew = joint[:len(y)]
dydxnew = joint[len(y):]
# set up chart
plot_xy([-width/2,width/2],[shift_joint,shift_joint],'gray')
t.penup()
t.goto(-50,3+shift_joint)
t.pendown()
t.write("y = 0 or dy/dx = 0")
plot_xy([shift_x,shift_x],[-width/2,width/2],'gray')
# plot data
plot_xy(xnew,ynew,'blue')
plot_xy(xnew,dydxnew,'red')